# Diebold-Li con filtro de Kalman y forecast

Este notebook replica en Python el flujo central del ejemplo de MathWorks sobre estimación y proyección del modelo de Diebold-Li con filtro de Kalman.

Referencia metodológica:
- https://it.mathworks.com/help/econ/using-the-kalman-filter-to-estimate-and-forecast-the-diebold-li-model.html

Adaptación al repo:
- usamos la base mensual derivada de `api_js/data/market_rates.csv`;
- trabajamos con la formulación Diebold-Li como un Nelson-Siegel dinámico;
- estimamos `lambda` por búsqueda en grilla;
- extraemos factores por OLS cross-sectional;
- estimamos la dinámica temporal con un `VAR(1)` por MCO;
- construimos un sistema de espacio de estados;
- aplicamos filtro y suavizado de Kalman;
- proyectamos curvas spot y curvas forward.

Observación importante:
esta implementación es seria y usable, pero no intenta replicar cada detalle de máxima verosimilitud del ejemplo de MathWorks. El objetivo aquí es dejar un flujo transparente, reproducible y fácil de extender dentro del proyecto.

## 1. Ecuaciones del modelo

La curva spot observada en el tiempo `t` y madurez `tau` se modela como:

$$
y_t(\tau) = \beta_{1,t} + \beta_{2,t}\left(\frac{1-e^{-\lambda \tau}}{\lambda \tau}\right) + \beta_{3,t}\left(\frac{1-e^{-\lambda \tau}}{\lambda \tau} - e^{-\lambda \tau}\right) + \varepsilon_t(\tau)
$$

donde:
- `beta_1,t` representa `level`;
- `beta_2,t` representa `slope`;
- `beta_3,t` representa `curvature`.

En forma de espacio de estados:

$$
y_t = \Lambda(\lambda) X_t + \varepsilon_t, \qquad \varepsilon_t \sim N(0, R)
$$

$$
X_t = c + A X_{t-1} + \eta_t, \qquad \eta_t \sim N(0, Q)
$$

donde `X_t = [level_t, slope_t, curvature_t]'` y `A` será estimada como un `VAR(1)` sobre los factores OLS.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import solve_discrete_lyapunov

plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.grid'] = True

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent.parent

DATA_PATH = ROOT / 'api_js' / 'data' / 'market_rates.csv'

RATE_MONTHS = {
    'TPM': 1,
    'SPC_03Y': 3,
    'SPC_06Y': 6,
    'SPC_1Y': 12,
    'SPC_2Y': 24,
    'SPC_3Y': 36,
    'SPC_4Y': 48,
    'SPC_5Y': 60,
    'SPC_10Y': 120,
}

COLUMNS = list(RATE_MONTHS.keys())
TAU_MONTHS = np.array([RATE_MONTHS[c] for c in COLUMNS], dtype=float)
TAU_YEARS = TAU_MONTHS / 12.0

print(DATA_PATH)
print(COLUMNS)

In [ ]:
def load_monthly_panel(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=['Date'])
    monthly = (
        df.set_index('Date')[COLUMNS]
        .resample('ME')
        .last()
        .dropna(how='all')
    )
    complete = monthly.dropna(subset=COLUMNS).copy()
    complete.index.name = 'Date'
    return complete


panel = load_monthly_panel(DATA_PATH)
panel.head(), panel.shape

## 2. Funciones base: loadings, OLS cross-sectional y `lambda`

El ejemplo de MathWorks parte con una etapa preliminar para obtener factores e inicializar el sistema. Aquí hacemos lo mismo con una búsqueda en grilla para `lambda` y luego una extracción OLS fecha a fecha.

In [ ]:
def ns_loadings(tau_years: np.ndarray, lambda_value: float) -> np.ndarray:
    slope = (1.0 - np.exp(-lambda_value * tau_years)) / (lambda_value * tau_years)
    curvature = slope - np.exp(-lambda_value * tau_years)
    return np.column_stack([np.ones_like(tau_years), slope, curvature])


def ols_betas_for_panel(yields_df: pd.DataFrame, lambda_value: float):
    X = ns_loadings(TAU_YEARS, lambda_value)
    XtX_inv = np.linalg.inv(X.T @ X)
    betas = []
    residuals = []
    rmse = []
    for _, row in yields_df.iterrows():
        y = row[COLUMNS].to_numpy(dtype=float)
        beta = XtX_inv @ X.T @ y
        fitted = X @ beta
        err = y - fitted
        betas.append(beta)
        residuals.append(err)
        rmse.append(np.sqrt(np.mean(err ** 2)))
    betas = pd.DataFrame(betas, index=yields_df.index, columns=['level', 'slope', 'curvature'])
    residuals = pd.DataFrame(residuals, index=yields_df.index, columns=COLUMNS)
    return betas, residuals, float(np.mean(rmse))


grid = np.linspace(0.02, 0.20, 120)
grid_scores = []
for lam in grid:
    _, _, avg_rmse = ols_betas_for_panel(panel, lam)
    grid_scores.append((lam, avg_rmse))

lambda_grid = pd.DataFrame(grid_scores, columns=['lambda', 'avg_rmse'])
best_lambda = float(lambda_grid.sort_values('avg_rmse').iloc[0]['lambda'])
best_lambda

In [ ]:
ols_betas, cross_residuals, avg_rmse = ols_betas_for_panel(panel, best_lambda)
avg_rmse

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(lambda_grid['lambda'], lambda_grid['avg_rmse'], color='tab:blue')
axes[0].axvline(best_lambda, color='tab:red', linestyle='--', label=f'lambda={best_lambda:.4f}')
axes[0].set_title('Búsqueda en grilla para lambda')
axes[0].set_xlabel('lambda')
axes[0].set_ylabel('RMSE promedio')
axes[0].legend()

ols_betas.plot(ax=axes[1])
axes[1].set_title('Factores OLS de Diebold-Li')
axes[1].set_xlabel('Fecha')
axes[1].set_ylabel('Factor')
plt.tight_layout()

## 3. Dinámica temporal: `VAR(1)` para los factores

Siguiendo la lógica estándar del modelo dinámico, estimamos:

$$
X_t = c + A X_{t-1} + \eta_t
$$

A partir de esta regresión obtenemos:
- `c`: intercepto del estado;
- `A`: matriz de transición;
- `Q`: covarianza de innovaciones del estado.

In [ ]:
def fit_var1(factors: pd.DataFrame):
    Y = factors.iloc[1:].to_numpy(dtype=float)
    Xlag = factors.iloc[:-1].to_numpy(dtype=float)
    X = np.column_stack([np.ones(len(Xlag)), Xlag])
    B = np.linalg.inv(X.T @ X) @ X.T @ Y
    c = B[0]
    A = B[1:].T
    resid = Y - X @ B
    Q = np.cov(resid.T, bias=False)
    return c, A, Q, resid


state_intercept, transition_matrix, state_cov, var_resid = fit_var1(ols_betas)
measurement_cov = np.diag(np.maximum(cross_residuals.var(axis=0, ddof=1).to_numpy(dtype=float), 1e-6))

state_intercept, transition_matrix, state_cov

## 4. Filtro y suavizado de Kalman

Usamos la matriz de medición fija `Lambda(lambda)` y las matrices estimadas `c`, `A`, `Q`, `R` para correr:
- filtro de Kalman hacia adelante;
- suavizado Rauch-Tung-Striebel hacia atrás.

La covarianza inicial del estado se fija con la solución de Lyapunov discreta:

$$
P_0 = A P_0 A' + Q
$$

y el estado inicial se toma como la media incondicional aproximada del `VAR(1)`.

## 4.1 ¿Qué hace exactamente el filtro de Kalman en este contexto?

En Diebold-Li dinámico, las tasas observadas no se ajustan directamente como una colección de curvas independientes. El punto central es que los factores `level`, `slope` y `curvature` se tratan como **estados latentes**: no se observan directamente, pero generan la cross-section de tasas en cada fecha.

El filtro de Kalman combina dos piezas de información:

1. **La ecuación de medición**

$$
y_t = \Lambda(\lambda) X_t + \varepsilon_t
$$

que dice cómo las tasas observadas se generan a partir de los factores.

2. **La ecuación de transición**

$$
X_t = c + A X_{t-1} + \eta_t
$$

que dice cómo esos factores evolucionan en el tiempo.

Operativamente, el filtro hace dos pasos en cada fecha `t`:

- **Predicción**: usa la dinámica temporal para anticipar el estado antes de ver las tasas del período.

$$
\hat X_{t|t-1} = c + A \hat X_{t-1|t-1}
$$

- **Actualización**: compara esa predicción con las tasas efectivamente observadas y corrige el estado usando la ganancia de Kalman.

$$
\hat X_{t|t} = \hat X_{t|t-1} + K_t\big(y_t - \Lambda \hat X_{t|t-1}\big)
$$

La ganancia `K_t` determina cuánto peso darle a:
- lo que venía diciendo la dinámica histórica del estado;
- lo que informan las tasas observadas en la fecha actual.

Si el error de medición `R` es alto, el filtro confía menos en la observación puntual. Si la observación es muy informativa, el filtro corrige más fuertemente.

## 4.2 ¿Cómo cambian los resultados frente a OLS fecha a fecha?

Cuando usamos solo OLS cross-sectional por fecha, cada curva se estima de manera aislada. Eso tiene una ventaja: el ajuste de cada fecha es simple y directo. Pero también tiene un costo: los factores pueden salir muy ruidosos y reaccionar demasiado a pequeñas anomalías de mercado.

Con Kalman, el sistema deja de tratar cada fecha como una isla. En vez de eso, fuerza coherencia temporal entre fechas consecutivas. Eso cambia los resultados de varias maneras:

1. **Factores más suaves**

Las series de `level`, `slope` y `curvature` dejan de presentar saltos artificiales tan marcados. El filtro interpreta parte de la variación como ruido de medición, no como cambio estructural real.

2. **Curvas más estables**

La curva ajustada en cada fecha suele verse menos errática y más consistente con la evolución general de la muestra.

3. **Menor sobreajuste a nodos puntuales**

Si una tasa observada en una madurez particular viene contaminada por iliquidez, microestructura o ruido transitorio, el filtro no necesariamente la absorbe por completo como señal verdadera.

4. **Mejor base para forecast**

Como los estados latentes estimados son menos ruidosos, la dinámica temporal posterior suele ser más útil para proyectar curvas futuras.

5. **Interpretación económica más limpia**

Los factores suavizados tienden a parecer más cercanos a movimientos estructurales de la curva y menos a ruido fecha a fecha.

## 4.3 Filtered versus smoothed

En este notebook calculamos dos versiones del estado:

- `filtered`: usa información disponible hasta la fecha `t`;
- `smoothed`: usa toda la muestra, incluyendo información futura, mediante el suavizado Rauch-Tung-Striebel.

Por eso la serie `smoothed` suele verse aún más estable que la `filtered`. Para análisis histórico y descomposición de factores, la versión suavizada suele ser la más útil. Para un contexto de estimación en tiempo real, la versión filtrada es la referencia natural.

## 4.4 Qué no hace mágicamente Kalman

El filtro de Kalman mejora la estimación dinámica, pero no corrige automáticamente cualquier problema de especificación. Si `lambda`, `A`, `Q` o `R` están mal calibrados, el sistema puede sobre-suavizar o sub-suavizar los factores. Por eso sigue siendo importante revisar:

- estabilidad de la dinámica temporal;
- magnitudes relativas de `Q` y `R`;
- calidad del ajuste cross-sectional;
- plausibilidad económica de los factores resultantes.

En síntesis, el filtro de Kalman no reemplaza el modelo: lo convierte en una estimación dinámica más robusta, menos ruidosa y más coherente con la evolución temporal de la curva.

In [ ]:
def kalman_filter_smoother(y_df, lambda_value, c, A, Q, R, x0=None, P0=None):
    H = ns_loadings(TAU_YEARS, lambda_value)
    Y = y_df[COLUMNS].to_numpy(dtype=float)
    nobs = Y.shape[0]
    nstate = A.shape[0]

    if x0 is None:
        x0 = np.linalg.solve(np.eye(nstate) - A, c)
    if P0 is None:
        P0 = solve_discrete_lyapunov(A, Q)

    x_pred = np.zeros((nobs, nstate))
    P_pred = np.zeros((nobs, nstate, nstate))
    x_filt = np.zeros((nobs, nstate))
    P_filt = np.zeros((nobs, nstate, nstate))

    x_prev = x0.copy()
    P_prev = P0.copy()

    for t in range(nobs):
        x_prior = c + A @ x_prev
        P_prior = A @ P_prev @ A.T + Q
        innovation = Y[t] - H @ x_prior
        S = H @ P_prior @ H.T + R
        K = P_prior @ H.T @ np.linalg.inv(S)
        x_post = x_prior + K @ innovation
        P_post = (np.eye(nstate) - K @ H) @ P_prior

        x_pred[t] = x_prior
        P_pred[t] = P_prior
        x_filt[t] = x_post
        P_filt[t] = P_post

        x_prev = x_post
        P_prev = P_post

    x_smooth = x_filt.copy()
    P_smooth = P_filt.copy()

    for t in range(nobs - 2, -1, -1):
        J = P_filt[t] @ A.T @ np.linalg.inv(P_pred[t + 1])
        x_smooth[t] = x_filt[t] + J @ (x_smooth[t + 1] - x_pred[t + 1])
        P_smooth[t] = P_filt[t] + J @ (P_smooth[t + 1] - P_pred[t + 1]) @ J.T

    return {
        'x_pred': x_pred,
        'P_pred': P_pred,
        'x_filt': x_filt,
        'P_filt': P_filt,
        'x_smooth': x_smooth,
        'P_smooth': P_smooth,
        'H': H,
    }


kf = kalman_filter_smoother(panel, best_lambda, state_intercept, transition_matrix, state_cov, measurement_cov)
smoothed_betas = pd.DataFrame(kf['x_smooth'], index=panel.index, columns=['level', 'slope', 'curvature'])
filtered_betas = pd.DataFrame(kf['x_filt'], index=panel.index, columns=['level', 'slope', 'curvature'])
smoothed_betas.head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for i, factor in enumerate(['level', 'slope', 'curvature']):
    axes[i].plot(ols_betas.index, ols_betas[factor], label='OLS', alpha=0.55)
    axes[i].plot(filtered_betas.index, filtered_betas[factor], label='Filtered', alpha=0.8)
    axes[i].plot(smoothed_betas.index, smoothed_betas[factor], label='Smoothed', linewidth=2)
    axes[i].set_title(factor)
    axes[i].legend(loc='best')
plt.tight_layout()

## 5. Curvas ajustadas y comparación con observadas

Reconstruimos la curva spot a partir de los factores suavizados:

$$
\hat y_t = \Lambda(\lambda) \hat X_t
$$

y comparamos contra la cross-section observada para una fecha reciente.

In [ ]:
def reconstruct_curve(months, beta_row, lambda_value):
    H = ns_loadings(np.asarray(months, dtype=float) / 12.0, lambda_value)
    return H @ np.asarray(beta_row, dtype=float)


last_date = panel.index[-1]
last_obs = panel.loc[last_date, COLUMNS].to_numpy(dtype=float)
last_ols = ols_betas.loc[last_date, ['level', 'slope', 'curvature']].to_numpy(dtype=float)
last_smoothed = smoothed_betas.loc[last_date, ['level', 'slope', 'curvature']].to_numpy(dtype=float)

curve_ols = reconstruct_curve(TAU_MONTHS, last_ols, best_lambda)
curve_smoothed = reconstruct_curve(TAU_MONTHS, last_smoothed, best_lambda)

plt.figure(figsize=(11, 5))
plt.plot(TAU_MONTHS, curve_ols, marker='o', label='Diebold-Li OLS')
plt.plot(TAU_MONTHS, curve_smoothed, marker='o', label='Kalman smoothed')
plt.scatter(TAU_MONTHS, last_obs, s=55, label='Observed', zorder=5)
plt.title(f'Curva spot en {last_date.date()}')
plt.xlabel('Madurez (meses)')
plt.ylabel('Tasa')
plt.legend()
plt.show()

## 6. Forecast del estado y de la curva

A partir del último estado suavizado proyectamos los factores con la ecuación de transición:

$$
E[X_{t+h}|\mathcal F_t] = c + A E[X_{t+h-1}|\mathcal F_t]
$$

y luego reconstruimos las curvas spot proyectadas.

También derivamos una curva forward discreta entre meses consecutivos:

$$
f(t_1, t_2) = \left(\frac{(1+z(t_2))^{t_2}}{(1+z(t_1))^{t_1}}\right)^{1/(t_2-t_1)} - 1
$$

con `t_1` y `t_2` medidos en años.

In [ ]:
def forecast_states(last_state, c, A, steps):
    states = []
    x = np.asarray(last_state, dtype=float).copy()
    for _ in range(steps):
        x = c + A @ x
        states.append(x.copy())
    return np.asarray(states)


def forward_curve_from_spot(months, spot_rates):
    months = np.asarray(months, dtype=float)
    spot = np.asarray(spot_rates, dtype=float) / 100.0
    forwards = []
    f_months = []
    for i in range(1, len(months)):
        t1 = months[i - 1] / 12.0
        t2 = months[i] / 12.0
        z1 = spot[i - 1]
        z2 = spot[i]
        f = ((1.0 + z2) ** t2 / (1.0 + z1) ** t1) ** (1.0 / (t2 - t1)) - 1.0
        forwards.append(f * 100.0)
        f_months.append(months[i])
    return np.asarray(f_months), np.asarray(forwards)


horizons = [1, 3, 6, 12]
future_states = forecast_states(last_smoothed, state_intercept, transition_matrix, max(horizons))
forecast_curves = {
    h: reconstruct_curve(TAU_MONTHS, future_states[h - 1], best_lambda)
    for h in horizons
}

plt.figure(figsize=(11, 5))
plt.plot(TAU_MONTHS, curve_smoothed, linewidth=2.4, label=f'Actual {last_date.date()}')
plt.scatter(TAU_MONTHS, last_obs, s=45, label='Observed')
for h in horizons:
    plt.plot(TAU_MONTHS, forecast_curves[h], marker='o', label=f'Forecast {h}M')
plt.title('Curvas spot proyectadas con Diebold-Li + Kalman')
plt.xlabel('Madurez (meses)')
plt.ylabel('Tasa')
plt.legend(ncol=2)
plt.show()

plt.figure(figsize=(11, 5))
fm, ff = forward_curve_from_spot(TAU_MONTHS, curve_smoothed)
plt.plot(fm, ff, linewidth=2.4, label=f'Actual {last_date.date()}')
for h in horizons:
    fm_h, ff_h = forward_curve_from_spot(TAU_MONTHS, forecast_curves[h])
    plt.plot(fm_h, ff_h, marker='o', label=f'Forecast {h}M')
plt.title('Curvas forward derivadas de las proyecciones spot')
plt.xlabel('Madurez forward (meses)')
plt.ylabel('Forward rate')
plt.legend(ncol=2)
plt.show()

## 7. Resumen operativo

Este notebook deja implementado el flujo completo:

1. cargar y agregar la base mensual;
2. calibrar `lambda` por grilla;
3. extraer factores `level`, `slope`, `curvature` por OLS;
4. estimar un `VAR(1)` para la dinámica del estado;
5. construir `R` y `Q` desde residuos;
6. correr filtro y suavizado de Kalman;
7. reconstruir curvas spot;
8. proyectar estados, curvas spot y curvas forward.

Extensiones naturales para una siguiente versión:
- estimar `lambda` por máxima verosimilitud conjunta;
- imponer restricciones de estabilidad sobre `A`;
- permitir `R` no diagonal;
- incorporar intervalos de confianza de forecast;
- comparar el ajuste con Svensson y con el módulo `Kalman` de la app web.